# 📈 BlueStock Mutual Fund Platform — Risk & Performance Analytics

This notebook delivers rigorous financial analytics across all 40 mutual fund schemes. It calculates **Daily Returns**, **Multi-Period CAGRs**, **Risk-Adjusted Ratios (Sharpe & Sortino)**, **OLS Regression Alpha & Beta vs NIFTY 100**, **Maximum Drawdowns with Peak-to-Trough Date Ranges**, a composite **0–100 Fund Scorecard**, and **Benchmark Tracking Error** comparisons.

---


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

PROCESSED_DIR = os.path.join('..', 'data', 'processed') if os.path.exists(os.path.join('..', 'data', 'processed')) else os.path.join('data', 'processed')

def p(name):
    for path in [os.path.join(PROCESSED_DIR, name), os.path.join('csv', name), name]:
        if os.path.exists(path): return path
    return name

print("Loading cleaned dataset and scorecard results...")
scorecard_df = pd.read_csv(p('fund_scorecard.csv'))
alpha_beta_df = pd.read_csv(p('alpha_beta.csv'))
nav_df = pd.read_csv(p('02_nav_history.csv'))
nav_df['date'] = pd.to_datetime(nav_df['date'])

bi_df = pd.read_csv(p('10_benchmark_indices.csv'))
bi_df['date'] = pd.to_datetime(bi_df['date'])

print(f"Loaded metrics for all {len(scorecard_df)} schemes!")


## 1. Top 10 Composite Fund Scorecard (0–100 Rating Scale)

The composite rating score synthesizes 5 core parameters:
- **30%**: 3-Year CAGR Rank
- **25%**: Risk-Adjusted Sharpe Ratio Rank
- **20%**: Alpha Outperformance Rank
- **15%**: Expense Ratio Rank (Inverse)
- **10%**: Max Drawdown Rank (Inverse)


In [ ]:
# Display Top 10 Rated Funds
top10_table = scorecard_df[['overall_rank', 'fund_score', 'scheme_name', 'fund_house', 'category', 'cagr_3yr_pct', 'sharpe_ratio', 'sortino_ratio', 'alpha_pct', 'max_drawdown_pct', 'expense_ratio_pct']].head(10)
display(top10_table)


## 2. Risk-Adjusted Performance (Sharpe vs. Sortino Ratios)

- **Sharpe Ratio**: Uses total volatility in the denominator ($R_f = 6.5\%$).
- **Sortino Ratio**: Uses downside volatility only, punishing only negative return days.


In [ ]:
# Bar Plot of Top 10 Schemes by Sharpe & Sortino Ratio
top10_ratios = scorecard_df.head(10).melt(id_vars=['scheme_name'], value_vars=['sharpe_ratio', 'sortino_ratio'], var_name='Metric', value_name='Ratio')

plt.figure(figsize=(14, 6))
sns.barplot(data=top10_ratios, x='scheme_name', y='Ratio', hue='Metric', palette='muted')
plt.title("Risk-Adjusted Ratios (Sharpe vs. Sortino) for Top 10 Rated Schemes", fontsize=14, fontweight="bold")
plt.xlabel("Scheme Name", fontsize=11)
plt.ylabel("Ratio Value", fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Risk Metric")
plt.tight_layout()
plt.show()


## 3. OLS Regression Analysis: Fund Alpha & Beta vs. NIFTY 100 Benchmark

Alpha measures manager stock-picking skill (annualized excess return over market line), while Beta measures systemic market exposure.


In [ ]:
# Scatter Plot of Alpha vs Beta across Funds
plt.figure(figsize=(12, 6))
sns.scatterplot(data=scorecard_df, x='beta', y='alpha_pct', hue='category', size='cagr_3yr_pct', sizes=(40, 300), palette='deep')
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='Zero Alpha Baseline')
plt.axvline(1, color='gray', linestyle=':', linewidth=1, label='Market Beta = 1.0')

plt.title("Scheme Alpha (%) vs. Beta (Systemic Market Risk)", fontsize=14, fontweight="bold")
plt.xlabel("Beta (Market Sensitivity)", fontsize=11)
plt.ylabel("Annualized Alpha (%)", fontsize=11)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 4. Maximum Drawdown & Downside Risk Profile

Maximum Drawdown measures peak-to-trough drop. The table highlights worst drawdown periods across equity and debt categories.


In [ ]:
# Display Worst Drawdown Risk Table
dd_table = scorecard_df[['scheme_name', 'category', 'max_drawdown_pct', 'worst_dd_period', 'cagr_3yr_pct']].sort_values('max_drawdown_pct', ascending=True).head(10)
display(dd_table)


## 5. Benchmark Comparison & Tracking Error Analysis

Top 5 ranked funds plotted against NIFTY 50 and NIFTY 100 benchmarks over 3 years.


In [ ]:
# Plot Normalized Performance of Top 5 Funds vs NIFTY 50 and NIFTY 100
top_5_codes = scorecard_df.head(5)['amfi_code'].tolist()
start_3y_date = nav_df['date'].max() - pd.DateOffset(years=3)

nifty50_df = bi_df[bi_df['index_name'] == 'NIFTY50'].sort_values('date').set_index('date')['close_value']
nifty100_df = bi_df[bi_df['index_name'] == 'NIFTY100'].sort_values('date').set_index('date')['close_value']

plt.figure(figsize=(14, 7))

n50_3y = nifty50_df[nifty50_df.index >= start_3y_date]
plt.plot(n50_3y.index, n50_3y / n50_3y.iloc[0] * 100, label="NIFTY 50 Benchmark", color="black", linestyle="--", linewidth=2.5)

n100_3y = nifty100_df[nifty100_df.index >= start_3y_date]
plt.plot(n100_3y.index, n100_3y / n100_3y.iloc[0] * 100, label="NIFTY 100 Benchmark", color="#444444", linestyle=":", linewidth=2.5)

for code in top_5_codes:
    s_name = scorecard_df[scorecard_df['amfi_code'] == code]['scheme_name'].values[0][:25]
    f_sub = nav_df[(nav_df['amfi_code'] == code) & (nav_df['date'] >= start_3y_date)].sort_values('date').set_index('date')['nav']
    plt.plot(f_sub.index, f_sub / f_sub.iloc[0] * 100, label=s_name, linewidth=2.0)

plt.title("Top 5 Rated Funds vs. NIFTY 50 & NIFTY 100 Benchmarks (3-Year Relative Performance)", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=11)
plt.ylabel("Normalized NAV / Index Level (Base = 100)", fontsize=11)
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()
